In [1]:
import numpy as np
import qiskit
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import Operator
from qiskit_aer import AerSimulator
from scipy.linalg import expm

# Problem Definition 1D Viscous Burgers' Equation
# PDE:     u_t + u*u_x = nu * u_xx
# Domain, grid, and boundary/initial conditions
L = 1.0
N = 4                     
dx = L / (N + 1)               
x_interior = np.array([(i + 1) * dx for i in range(N)])

nu = 0.05                      # viscosity
dt = 0.5                       # implicit time step
u_0 = 1.0
u_1 = 0.0

# Picard iteration
u_prev = (u_0 + (u_1 - u_0) * (x_interior / L)) + 0.1 * np.sin(np.pi * x_interior / L)

print("\n STEP 0: Burgers' Equation Setup (direct discretization)")
print(f"Domain: x in [0, {L}], nu = {nu}, dt = {dt}, dx = {dx}")
print(f"Boundary conditions on u: u(0,t) = {u_0}, u(L,t) = {u_1}")
print(f"Interior nodes x = {x_interior}")
print(f"Picard previous iterate u^n (interior) = {u_prev}")

# STEP 1: Build A and b directly from Burgers' equation using correct dimensionless numbers
Fo = nu * dt / dx ** 2         # Fourier number
CFL = u_prev * dt / (2 * dx)   # CFL number

A = np.zeros((N, N), dtype=complex)
b = np.zeros(N, dtype=complex)
for i in range(N):
    A[i, i] = 1 + 2 * Fo
    if i > 0:
        A[i, i - 1] = -(Fo + CFL[i])
    if i < N - 1:
        A[i, i + 1] = -(Fo - CFL[i])
    b[i] = u_prev[i]
b[0]  += (Fo + CFL[0]) * u_0  
b[-1] += (Fo - CFL[-1]) * u_1   

print("\n STEP 1: Linear System From Burgers Discretization")
print("Matrix A:")
print(A.real)
print("Hermitian?", np.allclose(A, A.conj().T), " <- expected False, this is the point")
print(f"Source vector b: {b.real}")

# STEP 1b: Hermitian Embedding
zero_block = np.zeros((N, N), dtype=complex)
H = np.block([[zero_block, A], [A.conj().T, zero_block]])
assert np.allclose(H, H.conj().T), "H must be Hermitian!"

dim_H = H.shape[0]
c_vec = np.zeros(dim_H, dtype=complex)
c_vec[:N] = b                      # c = [b; 0]  (block=0 half carries |b>)
norm_c = np.linalg.norm(c_vec)
state_c = c_vec / norm_c

print("\nSTEP 1b: Hermitian Dilation")
print("H is Hermitian:", np.allclose(H, H.conj().T))
print(f"Dilated state |c> = |b,0>/||.|| (dimension {dim_H}):")
print(state_c.real)

# STEP 2: Configure Quantum Registers & Phase Estimation Scaling
q_sys = QuantumRegister(3, name='sys')      # sys[0], sys[1] = node ; sys[2] = block
q_clock = QuantumRegister(2, name='clock')
q_ancilla = QuantumRegister(1, name='ancilla')
c_out = ClassicalRegister(4, name='meas')   # ancilla, block, node0, node1

circuit = QuantumCircuit(q_sys, q_clock, q_ancilla, c_out)

# Eigenvalues of H
eigvals_H = np.linalg.eigvalsh(H.real if np.allclose(H.imag, 0) else H)
sigma_max = np.abs(eigvals_H).max()

# Choose t so the LARGEST |eigenvalue|'s phase stays safely below the 0.5 threshold 
t = 0.8 * np.pi / sigma_max

# Ancilla rotation constant, capped by the smallest magnitude we can decode
C = None  # set after we know the decodable magnitudes (Step 5 setup below)

print("\nSTEP 2: Circuit Initialization")
print("Allocated 6 qubits (3 System [2 node + 1 block], 2 Clock, 1 Ancilla)")
print(f"Eigenvalues of H (= +-singular values of A): {np.sort(eigvals_H)}")
print(f"Time evolution parameter t = {t:.4f}")

# STEP 3: Quantum State Preparation
state_b_dil = (b / np.linalg.norm(b)).real
circuit.prepare_state(state_b_dil, [q_sys[0], q_sys[1]])
circuit.barrier()

# STEP 4: Quantum Phase Estimation (QPE) on H
circuit.h(q_clock)

U1_matrix = expm(1j * H * t)
U1_gate = Operator(U1_matrix).to_instruction().control(1)
circuit.append(U1_gate, [q_clock[0], q_sys[0], q_sys[1], q_sys[2]])

U2_matrix = expm(1j * H * t * 2)
U2_gate = Operator(U2_matrix).to_instruction().control(1)
circuit.append(U2_gate, [q_clock[1], q_sys[0], q_sys[1], q_sys[2]])

# Inverse QFT on the 2-qubit clock register
circuit.h(q_clock[0])
circuit.cp(-np.pi / 2, q_clock[1], q_clock[0])
circuit.h(q_clock[1])
circuit.swap(q_clock[0], q_clock[1])
circuit.barrier()

# STEP 5: Controlled Ancilla Rotation (SIGNED Eigenvalue Inversion)
n_clock = 2
decoded = {}
for j in range(1, 2 ** n_clock):
    lam = (j if j < 2 ** (n_clock - 1) else j - 2 ** n_clock) * (2 * np.pi) / (2 ** n_clock * t)
    decoded[j] = lam

C = 0.99 * min(abs(v) for v in decoded.values())
print("\nSTEP 5: Signed Eigenvalue Decoding")
for j, lam in decoded.items():
    print(f"  clock value j={j} ({format(j,'02b')}) -> decoded lambda = {lam:+.4f}")
print(f"Ancilla rotation constant C = {C:.4f}")

for clock_val, approx_lambda in decoded.items():
    theta = 2 * np.arcsin(np.clip(C / approx_lambda, -1.0, 1.0))

    binary_str = format(clock_val, '02b')
    for bit_idx, bit in enumerate(binary_str):
        if bit == '0':
            circuit.x(q_clock[1 - bit_idx])

    circuit.mcry(theta, q_clock[:], q_ancilla[0])

    for bit_idx, bit in enumerate(binary_str):
        if bit == '0':
            circuit.x(q_clock[1 - bit_idx])

circuit.barrier()

# STEP 6: Uncomputation
circuit.swap(q_clock[0], q_clock[1])
circuit.h(q_clock[1])
circuit.cp(np.pi / 2, q_clock[1], q_clock[0])
circuit.h(q_clock[0])

U2_adj_gate = Operator(U2_matrix.conj().T).to_instruction().control(1)
circuit.append(U2_adj_gate, [q_clock[1], q_sys[0], q_sys[1], q_sys[2]])

U1_adj_gate = Operator(U1_matrix.conj().T).to_instruction().control(1)
circuit.append(U1_adj_gate, [q_clock[0], q_sys[0], q_sys[1], q_sys[2]])

circuit.h(q_clock)
circuit.barrier()

# STEP 7: Measurement & Post-Selection
circuit.measure(q_ancilla[0], c_out[0])    
circuit.measure(q_sys[2], c_out[1])        # dilation block bit
circuit.measure(q_sys[0], c_out[2])        # node bit 0
circuit.measure(q_sys[1], c_out[3])        # node bit 1

simulator = AerSimulator()
compiled_circuit = qiskit.transpile(circuit, simulator)
shots = 200000
result = simulator.run(compiled_circuit, shots=shots).result()
counts = result.get_counts()

print("\n STEP 7: Quantum Execution Results (Raw Counts)")
print(f"Total Shots: {shots}")

successful_shots = 0
sys_counts = {'00': 0, '01': 0, '10': 0, '11': 0}

for bitstring, count in counts.items():
    ancilla_bit = bitstring[-1]
    block_bit = bitstring[-2]
    node_bits = bitstring[:-2]
    if ancilla_bit == '1' and block_bit == '1':
        successful_shots += count
        sys_counts[node_bits] += count

print(f"Successful Post-Selection Shots (ancilla=1 AND block=1): {successful_shots}")
print(f"Post-selection Probability: {successful_shots / shots:.4%}")

probabilities = np.array([sys_counts['00'], sys_counts['01'], sys_counts['10'], sys_counts['11']]) / successful_shots
amplitudes_hhl = np.sqrt(probabilities)

print("\nMeasured Normalized Quantum Amplitudes |u_HHL> (up to overall scale/sign):")
for idx, label in enumerate(['00', '01', '10', '11']):
    print(f"  |{label}>: Amplitude = {amplitudes_hhl[idx]:.4f} (Prob = {probabilities[idx]:.4f})")

# STEP 8: Classical Solution Comparison
u_classical = np.linalg.solve(A, b).real
scaling_factor = np.linalg.norm(u_classical)
u_hhl_reconstructed = amplitudes_hhl * scaling_factor

print("\nFINAL SOLUTION COMPARISON  (direct Burgers Picard-linear solve)\n")
print(f"{'Node (k)':<10} | {'x':<6} | {'u_HHL':<15} | {'u_exact':<15}")
print("-" * 60)
for k in range(4):
    print(f"k = {k + 1:<6} | {x_interior[k]:<6.2f} | {u_hhl_reconstructed[k]:<15.4f} | {u_classical[k]:<15.4f}")


 STEP 0: Burgers' Equation Setup (direct discretization)
Domain: x in [0, 1.0], nu = 0.05, dt = 0.5, dx = 0.2
Boundary conditions on u: u(0,t) = 1.0, u(L,t) = 0.0
Interior nodes x = [0.2 0.4 0.6 0.8]
Picard previous iterate u^n (interior) = [0.85877853 0.69510565 0.49510565 0.25877853]

 STEP 1: Linear System From Burgers Discretization
Matrix A:
[[ 2.25        0.44847316  0.          0.        ]
 [-1.49388206  2.25        0.24388206  0.        ]
 [ 0.         -1.24388206  2.25       -0.00611794]
 [ 0.          0.         -0.94847316  2.25      ]]
Hermitian? False  <- expected False, this is the point
Source vector b: [2.55725168 0.69510565 0.49510565 0.25877853]

STEP 1b: Hermitian Dilation
H is Hermitian: True
Dilated state |c> = |b,0>/||.|| (dimension 8):
[0.94423309 0.25665904 0.18281155 0.09555072 0.         0.
 0.         0.        ]

STEP 2: Circuit Initialization
Allocated 6 qubits (3 System [2 node + 1 block], 2 Clock, 1 Ancilla)
Eigenvalues of H (= +-singular values of A): [


 STEP 7: Quantum Execution Results (Raw Counts)
Total Shots: 200000
Successful Post-Selection Shots (ancilla=1 AND block=1): 123707
Post-selection Probability: 61.8535%

Measured Normalized Quantum Amplitudes |u_HHL> (up to overall scale/sign):
  |00>: Amplitude = 0.6687 (Prob = 0.4472)
  |01>: Amplitude = 0.6098 (Prob = 0.3719)
  |10>: Amplitude = 0.3861 (Prob = 0.1491)
  |11>: Amplitude = 0.1784 (Prob = 0.0318)

FINAL SOLUTION COMPARISON  (direct Burgers Picard-linear solve)

Node (k)   | x      | u_HHL           | u_exact        
------------------------------------------------------------
k = 1      | 0.20   | 1.0254          | 0.9628         
k = 2      | 0.40   | 0.9350          | 0.8719         
k = 3      | 0.60   | 0.5920          | 0.7032         
k = 4      | 0.80   | 0.2735          | 0.4114         
